# Phân loại cảm xúc đánh giá Điện Máy Xanh

| Bước | Nơi làm | Đầu ra |
|---|---|---|
| B1 Cào dữ liệu | `crawl_reviews.py` | `data/01.raw_reviews.csv` |
| B2 Làm sạch nhẹ | `clean_reviews.py` | `data/02.clean_reviews.csv` |
| B3 Gán nhãn | Người gán tay 500 câu, Claude gán toàn bộ theo `prompt_ai_label1.md` | `data/04.human_labels.xlsx`, `data/05.claude_v2.csv` |
| B3–B7 | **notebook này** | đọc nhãn → EDA → chia tập → tiền xử lý → đặc trưng → huấn luyện → đánh giá |

Nhãn: `-1` tiêu cực, `0` trung tính, `1` tích cực.

## Thiết lập thư viện

In [ ]:
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from underthesea import word_tokenize
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

pd.set_option('display.max_colwidth', 120)
DATA, OUT = Path('data'), Path('out')
OUT.mkdir(exist_ok=True)
SEED = 42
LABELS, NAMES = [-1, 0, 1], ['Tiêu cực', 'Trung tính', 'Tích cực']
MAU = ['#d9534f', '#f0ad4e', '#5cb85c']  # màu của từng nhãn trong các biểu đồ

## 1. Dữ liệu và nhãn (B3)

Nhãn theo số sao có sẵn khi cào nhưng bị nhiễu: nhiều review 3 sao thực chất đang chê. Vì vậy nhóm gán lại theo **nội dung chữ**:

- `04.human_labels.xlsx`: nhóm gán tay 500 câu, **dùng làm tập test chuẩn**
- `05.claude_v2.csv`: Claude gán **toàn bộ** câu theo `prompt_ai_label1.md` (luật rút ra từ cách nhóm gán tay), **dùng làm nhãn huấn luyện**

In [ ]:
# Dữ liệu đã làm sạch. nhan_sao là nhãn suy từ số sao lúc cào, chỉ dùng để so sánh.
df = pd.read_csv(DATA / '02.clean_reviews.csv')[['review_id', 'product_id', 'so_sao', 'noi_dung', 'nhom_cam_xuc']]
df['nhan_sao'] = df['nhom_cam_xuc'].map(dict(zip(NAMES, LABELS)))

# Nhãn AI cho toàn bộ câu: dùng để huấn luyện.
ai = pd.read_csv(DATA / '05.claude_v2.csv').set_index('review_id')['label']
assert ai.index.is_unique and set(ai.index) == set(df['review_id']), \
    '05 phải có đủ mọi câu của 02, mỗi câu một dòng (cào lại dữ liệu thì phải gán lại)'
assert ai.isin(LABELS).all(), '05: nhãn chỉ được là -1, 0, 1'
df['label'] = df['review_id'].map(ai)

# Nhãn người cho 500 câu: dùng làm tập test.
nguoi = pd.read_excel(DATA / '04.human_labels.xlsx').set_index('review_id')['label']
assert nguoi.notna().all(), f'04: còn {nguoi.isna().sum()} câu chưa gán nhãn'
nguoi = nguoi.astype(int)
assert nguoi.isin(LABELS).all() and nguoi.index.isin(df['review_id']).all()

print(f'{len(df):,} câu từ {df["product_id"].nunique()} sản phẩm | nhãn AI: {len(ai):,} câu | nhãn người: {len(nguoi)} câu')

### Chất lượng nhãn: so với nhãn người (tỉ lệ khớp và Cohen's Kappa)

In [ ]:
sao = df.set_index('review_id')['nhan_sao']

def so_voi_nguoi(nguon, nhan):
    nhan = nhan[nguoi.index]
    return {'Nguồn nhãn': nguon, 'Tỉ lệ khớp': (nhan == nguoi).mean(),
            "Cohen's Kappa": cohen_kappa_score(nguoi, nhan)}

display(pd.DataFrame([so_voi_nguoi('Số sao', sao), so_voi_nguoi('AI (Claude)', ai)]).round(3))
print('Lưu ý: luật trong prompt rút ra từ chính các câu này, nên độ khớp của nhãn AI ở đây cao hơn thực tế.')

### Các câu AI gán khác người

In [ ]:
lech = pd.DataFrame({'noi_dung': df.set_index('review_id')['noi_dung'][nguoi.index],
                     'nguoi': nguoi, 'ai': ai[nguoi.index]}).query('nguoi != ai')
print(f'{len(lech)}/{len(nguoi)} câu AI gán khác người')
lech

## 2. Khám phá dữ liệu (EDA)

Phân tích trên **toàn bộ** dữ liệu đã gán nhãn AI, trước khi chia tập.

### 2.1 Tổng quan

In [ ]:
print('Kích thước:', df.shape)
print('Ô trống:', int(df[['noi_dung', 'label']].isna().sum().sum()),
      '| nội dung trùng:', int(df['noi_dung'].duplicated().sum()),
      '| review_id trùng:', int(df['review_id'].duplicated().sum()))
df[['noi_dung', 'so_sao', 'label']].sample(8, random_state=SEED)

### 2.2 Phân bố nhãn
Lớp trung tính ít hơn hẳn, nên các mô hình đặt `class_weight='balanced'` và được so bằng Macro F1.

In [ ]:
dem = df['label'].value_counts().reindex(LABELS)
display(pd.DataFrame({'Số câu': dem.values, 'Tỉ lệ (%)': (dem / len(df) * 100).round(1).values}, index=NAMES))
ax = dem.set_axis(NAMES).plot(kind='bar', rot=0, color=MAU, figsize=(6, 3.5))
ax.bar_label(ax.containers[0])
ax.set_title('Phân bố nhãn AI (toàn bộ dữ liệu)'); ax.set_ylabel('Số câu')
plt.tight_layout(); plt.show()

### 2.3 Nhãn AI so với số sao
Mỗi hàng là một nhóm số sao, ô là tỉ lệ (%) câu trong nhóm đó được gán từng nhãn.

In [ ]:
ti_le = pd.crosstab(df['so_sao'], df['label'], normalize='index').reindex(columns=LABELS) * 100
fig, ax = plt.subplots(figsize=(6, 3.8))
im = ax.imshow(ti_le.values, cmap='Blues', vmin=0, vmax=100)
ax.set_xticks(range(len(NAMES)), NAMES)
ax.set_yticks(range(len(ti_le)), [f'{s} sao' for s in ti_le.index])
for (i, j), v in np.ndenumerate(ti_le.values):
    ax.text(j, i, f'{v:.0f}%', ha='center', va='center', color='white' if v > 60 else 'black')
ax.set_title('Nhãn AI theo số sao'); fig.colorbar(im, ax=ax, label='%')
plt.tight_layout(); plt.show()
print(f'Nhãn AI khớp nhãn theo số sao: {(df["label"] == df["nhan_sao"]).mean():.1%}')

### 2.4 Độ dài review
Câu chê thường dài hơn vì người viết hay kể lại lỗi và quá trình bảo hành.

In [ ]:
df['so_tu'] = df['noi_dung'].str.split().str.len()
display(df.groupby('label')['so_tu'].describe().round(1).set_axis(NAMES))
so_tu_theo_nhan = [df.loc[df['label'] == c, 'so_tu'] for c in LABELS]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for so_tu, ten, mau in zip(so_tu_theo_nhan, NAMES, MAU):
    ax[0].hist(so_tu.clip(upper=100), bins=40, alpha=0.55, label=ten, color=mau)
ax[0].set_title('Phân bố số từ (cắt ở 100)'); ax[0].set_xlabel('Số từ'); ax[0].legend()
ax[1].boxplot(so_tu_theo_nhan, showfliers=False)
ax[1].set_xticks([1, 2, 3], NAMES); ax[1].set_title('Số từ theo nhãn (bỏ điểm ngoại lai)')
plt.tight_layout(); plt.show()

### 2.5 Phân bố nhãn theo sản phẩm

In [ ]:
sp = pd.crosstab(df['product_id'], df['label']).reindex(columns=LABELS).set_axis(NAMES, axis=1)
sp = sp.loc[sp.sum(axis=1).sort_values(ascending=False).index]
ax = sp.plot(kind='barh', stacked=True, figsize=(9, 5), color=MAU)
ax.invert_yaxis(); ax.set_xlabel('Số câu'); ax.set_title('Số review mỗi sản phẩm theo nhãn')
plt.tight_layout(); plt.show()

### 2.6 Cụm 2 âm tiết hay gặp nhất theo nhãn
Tiếng Việt nhiều từ ghép 2 âm tiết ("hài lòng", "bảo hành"), nên đếm cụm 2 âm tiết liền nhau cho dễ đọc hơn đếm từng âm tiết.

In [ ]:
def cum_pho_bien(cac_cau, n=12):
    # Đếm cụm 2 âm tiết liền nhau, trả về n cụm nhiều nhất
    dem = CountVectorizer(ngram_range=(2, 2), token_pattern=r'(?u)\b\w+\b')
    tong = np.asarray(dem.fit_transform(cac_cau.str.lower()).sum(axis=0)).ravel()
    top = np.argsort(-tong)[:n]
    return pd.Series(tong[top], index=dem.get_feature_names_out()[top])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, c, ten, mau in zip(axes, LABELS, NAMES, MAU):
    cum_pho_bien(df.loc[df['label'] == c, 'noi_dung'])[::-1].plot(kind='barh', ax=ax, color=mau)
    ax.set_title(ten)
plt.tight_layout(); plt.show()

## 3. Chia tập dữ liệu
- **Test**: các câu nhóm gán tay, dùng **nhãn người**
- **Validation**: 15% dữ liệu, dùng để chọn mức từ dừng và chọn mô hình
- **Train**: phần còn lại, dùng **nhãn AI**

Mỗi câu chỉ nằm trong một tập. Tập test chỉ dùng một lần ở cuối.

In [ ]:
test = df[df['review_id'].isin(nguoi.index)].copy()
test['label'] = test['review_id'].map(nguoi)          # test dùng nhãn người
con_lai = df[~df['review_id'].isin(nguoi.index)]      # train và validation dùng nhãn AI
train, val = train_test_split(con_lai, test_size=round(0.15 * len(df)), stratify=con_lai['label'], random_state=SEED)
train, val = train.copy(), val.copy()
pd.DataFrame({ten: d['label'].value_counts().reindex(LABELS) for ten, d in
              [('Train', train), ('Validation', val), ('Test', test)]}).set_axis(NAMES)

## 4. Tiền xử lý (B4)
1. **Chữ thường** và **dịch teencode** theo `data/teencode.csv` ("ko" thành "không"). Đơn vị đứng sau số được giữ nguyên ("50 k").
2. **Tách từ tiếng Việt** bằng `underthesea` ("sản_phẩm", "hài_lòng") và bỏ token chỉ gồm dấu câu.
3. **Từ dừng**: kho chung `tu_dung.txt` chứa cả những từ mang cảm xúc ("không", "rất", "quá"), nên không bỏ hết mà kiểm tra trên train trước.

In [ ]:
teen = pd.read_csv(DATA / 'teencode.csv', dtype=str, keep_default_na=False)
TEENCODE = dict(zip(teen['viet_tat'].str.strip(), teen['day_du'].str.strip()))
TU_DUNG = frozenset(l.strip().replace(' ', '_') for l in open(DATA / 'tu_dung.txt', encoding='utf-8')
                    if l.strip() and not l.startswith('#'))

def tach_tu(cau):
    cau = str(cau).lower()
    doi = lambda m: m[0] if m[0] not in TEENCODE or re.search(r'\d\s*$', cau[:m.start()]) else TEENCODE[m[0]]
    cau = re.sub(r'\w+', doi, cau)
    return ' '.join(t for t in word_tokenize(cau, format='text').split() if re.search(r'\w', t))

for d in (train, val, test):
    d['tach_tu'] = d['noi_dung'].apply(tach_tu)
print(f'Kho từ dừng: {len(TU_DUNG):,} từ')
train[['noi_dung', 'tach_tu']].head(5)

### Từ dừng nào mang cảm xúc? Phân tích độ lệch lớp trên train
$$\text{Độ lệch}(w, c) = \frac{P(c \mid \text{câu có } w)}{P(c)}$$
Từ dừng xuất hiện trong ít nhất 3 câu và có độ lệch ≥ 1.3 với một lớp nào đó thì **được giữ lại**.

In [ ]:
p_lop = train['label'].value_counts(normalize=True)
dem = {c: Counter() for c in LABELS}
for c, cau in zip(train['label'], train['tach_tu']):
    dem[c].update(set(cau.split()) & TU_DUNG)

rows = []
for w in set().union(*dem.values()):
    n = sum(dem[c][w] for c in LABELS)
    if n >= 3:
        lech = {c: dem[c][w] / n / p_lop[c] for c in LABELS}
        c_max = max(lech, key=lech.get)
        rows.append({'Từ dừng': w, 'Số câu': n, 'Lệch về': NAMES[LABELS.index(c_max)], 'Độ lệch': round(lech[c_max], 2)})
bang_lech = pd.DataFrame(rows).sort_values('Độ lệch', ascending=False)
GIU_LAI = set(bang_lech.loc[bang_lech['Độ lệch'] >= 1.3, 'Từ dừng'])
print(f'Giữ lại {len(GIU_LAI)} từ dừng mang cảm xúc')
bang_lech.head(12)

### So sánh 3 mức từ dừng trên validation (TF-IDF + LinearSVC)

In [ ]:
def bo_tu_dung(cau, muc):
    if muc == 'khong':
        return cau
    return ' '.join(t for t in cau.split() if t not in TU_DUNG or (muc == 'loc' and t in GIU_LAI))

MUC = {'khong': 'Không bỏ từ dừng', 'nguyen': 'Bỏ nguyên kho', 'loc': 'Bỏ kho, giữ từ cảm xúc'}
kq_muc = []
for muc, mo_ta in MUC.items():
    x_tr, x_va = train['tach_tu'].apply(bo_tu_dung, muc=muc), val['tach_tu'].apply(bo_tu_dung, muc=muc)
    vec = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False, sublinear_tf=True)
    clf = LinearSVC(class_weight='balanced', random_state=SEED).fit(vec.fit_transform(x_tr), train['label'])
    pred = clf.predict(vec.transform(x_va))
    kq_muc.append({'Mức': muc, 'Chiến lược': mo_ta, 'Số từ vựng': len(vec.vocabulary_),
                   'Số từ TB/câu': round(x_tr.str.split().str.len().mean(), 1),
                   'Macro F1 (val)': round(f1_score(val['label'], pred, average='macro'), 4)})
kq_muc = pd.DataFrame(kq_muc)
MUC_CHON = kq_muc.loc[kq_muc['Macro F1 (val)'].idxmax(), 'Mức']
for d in (train, val, test):
    d['van_ban'] = d['tach_tu'].apply(bo_tu_dung, muc=MUC_CHON)
print('Chọn mức:', MUC[MUC_CHON])
kq_muc

## 5. Trích chọn đặc trưng (B5)
Mọi bộ biến đổi đều **chỉ fit trên train**, sau đó transform validation và test.
1. **BoW**: đếm unigram và bigram, bỏ từ chỉ xuất hiện 1 lần (`min_df=2`)
2. **TF-IDF**: như BoW nhưng hạ trọng số những từ câu nào cũng có, dùng $1+\log(tf)$
3. **Word2Vec**: vector ngữ nghĩa dày 100 chiều. Vector câu là trung bình vector các từ trong câu

In [ ]:
X, y = {}, {'train': train['label'], 'val': val['label'], 'test': test['label']}
cau_hinh = dict(tokenizer=str.split, token_pattern=None, lowercase=False, ngram_range=(1, 2), min_df=2)
VECTORIZERS = {'BoW': CountVectorizer(**cau_hinh), 'TF-IDF': TfidfVectorizer(**cau_hinh, sublinear_tf=True)}
for ten, vec in VECTORIZERS.items():
    X[ten] = {'train': vec.fit_transform(train['van_ban']),   # chỉ fit trên train
              'val': vec.transform(val['van_ban']), 'test': vec.transform(test['van_ban'])}
    print(f'{ten}: {X[ten]["train"].shape[1]:,} đặc trưng')

# Ví dụ: các thành phần khác 0 trong vector TF-IDF của một câu
vi_du = 3
hang = X['TF-IDF']['train'][vi_du]
ten_dac_trung = VECTORIZERS['TF-IDF'].get_feature_names_out()
print('\nCâu:', train['van_ban'].iloc[vi_du])
print('TF-IDF:', {ten_dac_trung[j]: round(float(x), 3) for j, x in zip(hang.indices, hang.data)})

### Word2Vec (skip-gram + negative sampling)
gensim chưa có bản cài cho Python 3.14 trên Windows, nên nhóm tự cài đặt bằng numpy. Với mỗi cặp (từ $w$, từ ngữ cảnh $c$) cách nhau tối đa 5 từ, thuật toán tối đa hóa
$$\log\sigma(\mathbf{v}_w\cdot\mathbf{u}_c) + \sum_{k=1}^{5}\log\sigma(-\mathbf{v}_w\cdot\mathbf{u}_k)$$
trong đó $u_k$ là các từ âm, được lấy ngẫu nhiên theo tần suất mũ 3/4.

In [ ]:
def train_word2vec(cau, dim=100, window=5, min_count=2, neg=5, epochs=30, lr=0.025, batch=512, seed=SEED):
    rng = np.random.default_rng(seed)
    dem = Counter(w for c in cau for w in c)
    vocab = {w: i for i, w in enumerate(sorted(w for w, n in dem.items() if n >= min_count))}
    cap = np.array([(ids[i], ids[j]) for c in cau for ids in [[vocab[w] for w in c if w in vocab]]
                    for i in range(len(ids)) for j in range(max(0, i - window), min(len(ids), i + window + 1)) if j != i])
    p_am = np.array([dem[w] for w in vocab], dtype=float) ** 0.75
    p_am /= p_am.sum()
    W, U = (rng.random((len(vocab), dim)) - 0.5) / dim, np.zeros((len(vocab), dim))
    sig = lambda x: 1 / (1 + np.exp(-np.clip(x, -10, 10)))
    tong, buoc = epochs * int(np.ceil(len(cap) / batch)), 0
    for _ in range(epochs):
        thu_tu = rng.permutation(len(cap))
        for s in range(0, len(cap), batch):
            a = lr * max(1e-4, 1 - buoc / tong); buoc += 1
            w, c = cap[thu_tu[s:s + batch]].T
            am = rng.choice(len(vocab), size=(len(w), neg), p=p_am)
            g_d = sig((W[w] * U[c]).sum(1)) - 1                    # gradient cặp đúng
            g_a = sig(np.einsum('bkd,bd->bk', U[am], W[w]))         # gradient mẫu âm
            grad_w = g_d[:, None] * U[c] + np.einsum('bk,bkd->bd', g_a, U[am])
            np.add.at(U, c, -a * g_d[:, None] * W[w])
            np.add.at(U, am.ravel(), (-a * g_a[:, :, None] * W[w][:, None, :]).reshape(-1, dim))
            np.add.at(W, w, -a * grad_w)
    return vocab, W

vocab, W2V = train_word2vec([s.split() for s in train['van_ban']])

def vector_cau(cau):
    ids = [vocab[w] for w in cau.split() if w in vocab]
    return W2V[ids].mean(axis=0) if ids else np.zeros(W2V.shape[1])

X['Word2Vec'] = {k: np.vstack(d['van_ban'].apply(vector_cau)) for k, d in [('train', train), ('val', val), ('test', test)]}
print(f'Word2Vec: {len(vocab):,} từ, vector câu {X["Word2Vec"]["train"].shape[1]} chiều')

In [ ]:
def tu_gan_nghia(tu, n=8):
    M = W2V / (np.linalg.norm(W2V, axis=1, keepdims=True) + 1e-9)
    sim, ten = M @ M[vocab[tu]], list(vocab)
    return [(ten[j], round(float(sim[j]), 2)) for j in np.argsort(-sim)[1:n + 1]]

for tu in ['tốt', 'lỗi']:
    if tu in vocab:
        print(tu, '→', tu_gan_nghia(tu))

## 6. Huấn luyện mô hình
- **MultinomialNB**: hợp với đặc trưng đếm hoặc trọng số không âm (BoW, TF-IDF). Không dùng được với Word2Vec vì vector có giá trị âm
- **LinearSVC** và **Logistic Regression**: mô hình tuyến tính, đặt `class_weight='balanced'` để bù cho lớp trung tính ít mẫu

Mọi mô hình học trên train. Mô hình tốt nhất được **chọn theo Macro F1 trên validation**, sau đó mới xem điểm trên test.

In [ ]:
MODELS = {
    'MultinomialNB': MultinomialNB(),
    'LinearSVC': LinearSVC(class_weight='balanced', max_iter=5000, random_state=SEED),
    'LogReg': LogisticRegression(class_weight='balanced', max_iter=3000, random_state=SEED),
}
# MultinomialNB không dùng được với Word2Vec vì vector có giá trị âm
experiments = [(f, m) for f in ['BoW', 'TF-IDF'] for m in MODELS] + [('Word2Vec', 'LinearSVC'), ('Word2Vec', 'LogReg')]

def tinh_diem(y_true, y_pred):
    f1_tung_lop = f1_score(y_true, y_pred, average=None, labels=LABELS)
    return {'Accuracy': accuracy_score(y_true, y_pred), 'Macro F1': f1_score(y_true, y_pred, average='macro'),
            **{f'F1 {ten}': f for ten, f in zip(NAMES, f1_tung_lop)}}

def ghi_ket_qua(feature, model, p_val, p_test):
    results.append({'Feature': feature, 'Model': model, 'Val Macro F1': f1_score(y['val'], p_val, average='macro'),
                    **tinh_diem(y['test'], p_test)})

results, predictions = [], {}
lop_nhieu = train['label'].mode()[0]
ghi_ket_qua('(baseline)', 'Luôn đoán lớp nhiều nhất', np.full(len(val), lop_nhieu), np.full(len(test), lop_nhieu))
ghi_ket_qua('(baseline)', 'Nhãn theo số sao', val['nhan_sao'], test['nhan_sao'])

for feat, algo in experiments:
    clf = clone(MODELS[algo]).fit(X[feat]['train'], y['train'])
    predictions[(feat, algo)] = clf.predict(X[feat]['test'])
    ghi_ket_qua(feat, algo, clf.predict(X[feat]['val']), predictions[(feat, algo)])

## 7. Đánh giá
Các cột từ `Accuracy` trở đi là điểm trên **test**. Bảng được sắp xếp theo Macro F1 trên validation.

In [ ]:
scores = pd.DataFrame(results).sort_values('Val Macro F1', ascending=False).round(4).reset_index(drop=True)
scores

In [ ]:
ve = scores[scores['Feature'] != '(baseline)']
ve = ve.set_index(ve['Feature'] + ' + ' + ve['Model'])
ve[['Val Macro F1', 'Macro F1']].rename(columns={'Macro F1': 'Test Macro F1'}).plot(kind='bar', figsize=(10, 4), rot=30)
plt.ylim(0, 1); plt.title('Macro F1: validation và test'); plt.tight_layout(); plt.show()

### Mô hình tốt nhất (chọn theo validation): ma trận nhầm lẫn và báo cáo trên test

In [ ]:
best = tuple(scores.loc[scores['Feature'] != '(baseline)'].iloc[0][['Feature', 'Model']])
print('Mô hình chọn:', ' + '.join(best))
ConfusionMatrixDisplay(confusion_matrix(y['test'], predictions[best], labels=LABELS), display_labels=NAMES).plot(cmap='Blues')
plt.title('Confusion matrix (test): ' + ' + '.join(best)); plt.show()
print(classification_report(y['test'], predictions[best], labels=LABELS, target_names=NAMES, digits=4, zero_division=0))

### Phân tích lỗi

In [ ]:
sai = test[['noi_dung', 'so_sao', 'label']].assign(du_doan=predictions[best]).query('label != du_doan')
print(f'{len(sai)}/{len(test)} câu đoán sai')
sai.head(12)

In [ ]:
scores.to_csv(OUT / 'model_scores.csv', index=False, encoding='utf-8-sig')
print('Đã lưu out/model_scores.csv')